# Exploratory Data Analysis
## AI Lab ML Service - Dataset Analysis

This notebook explores the sentiment and keyword extraction datasets.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Random seed
np.random.seed(42)

## 1. Load Datasets

In [ ]:
# Load sentiment data
sentiment_train = pd.read_csv('../data/sentiment_train.csv')
sentiment_val = pd.read_csv('../data/sentiment_val.csv')
sentiment_test = pd.read_csv('../data/sentiment_test.csv')

print("Sentiment Dataset:")
print(f"  Train: {len(sentiment_train)} samples")
print(f"  Val:   {len(sentiment_val)} samples")
print(f"  Test:  {len(sentiment_test)} samples")
print(f"  Total: {len(sentiment_train) + len(sentiment_val) + len(sentiment_test)} samples")

In [ ]:
# Load keyword data
keyword_train = pd.read_csv('../data/keywords_train.csv')
keyword_val = pd.read_csv('../data/keywords_val.csv')
keyword_test = pd.read_csv('../data/keywords_test.csv')

print("Keyword Dataset:")
print(f"  Train: {len(keyword_train)} samples")
print(f"  Val:   {len(keyword_val)} samples")
print(f"  Test:  {len(keyword_test)} samples")
print(f"  Total: {len(keyword_train) + len(keyword_val) + len(keyword_test)} samples")

## 2. Sentiment Dataset Analysis

In [ ]:
# Display first few samples
print("Sample data:")
sentiment_train.head()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (df, split) in enumerate([(sentiment_train, 'Train'), 
                                     (sentiment_val, 'Validation'), 
                                     (sentiment_test, 'Test')]):
    counts = df['label_name'].value_counts()
    axes[idx].bar(counts.index, counts.values, color=['#e74c3c', '#95a5a6', '#2ecc71'])
    axes[idx].set_title(f'{split} Set - Class Distribution')
    axes[idx].set_xlabel('Sentiment')
    axes[idx].set_ylabel('Count')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Print statistics
print("\nClass distribution (Train):")
print(sentiment_train['label_name'].value_counts())
print(f"\nClass balance:")
print(sentiment_train['label_name'].value_counts(normalize=True))

In [ ]:
# Text length analysis
sentiment_train['text_length'] = sentiment_train['text'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(sentiment_train['text_length'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Text Length (words)')
plt.ylabel('Frequency')
plt.title('Distribution of Text Lengths')
plt.axvline(sentiment_train['text_length'].mean(), color='red', linestyle='--', label='Mean')
plt.legend()

plt.subplot(1, 2, 2)
sentiment_train.boxplot(column='text_length', by='label_name', figsize=(8, 5))
plt.xlabel('Sentiment')
plt.ylabel('Text Length (words)')
plt.title('Text Length by Sentiment')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f"\nText length statistics:")
print(f"  Mean: {sentiment_train['text_length'].mean():.2f} words")
print(f"  Median: {sentiment_train['text_length'].median():.2f} words")
print(f"  Min: {sentiment_train['text_length'].min()} words")
print(f"  Max: {sentiment_train['text_length'].max()} words")

In [ ]:
# Sample texts from each class
print("Sample texts from each sentiment class:\n")
for label in ['positive', 'neutral', 'negative']:
    print(f"=== {label.upper()} ===")
    samples = sentiment_train[sentiment_train['label_name'] == label]['text'].sample(2, random_state=42)
    for i, text in enumerate(samples, 1):
        print(f"{i}. {text}")
    print()

## 3. Keyword Dataset Analysis

In [ ]:
# Display first few samples
print("Sample data:")
keyword_train.head()

In [ ]:
# Parse keywords from string
import ast

def parse_keywords(kw_str):
    try:
        return ast.literal_eval(kw_str)
    except:
        return []

keyword_train['keywords_list'] = keyword_train['keywords'].apply(parse_keywords)
keyword_train['num_keywords'] = keyword_train['keywords_list'].apply(len)
keyword_train['doc_length'] = keyword_train['document'].apply(lambda x: len(str(x).split()))

# Statistics
print("Keyword Dataset Statistics:")
print(f"  Avg keywords per document: {keyword_train['num_keywords'].mean():.2f}")
print(f"  Avg document length: {keyword_train['doc_length'].mean():.2f} words")
print(f"  Min keywords: {keyword_train['num_keywords'].min()}")
print(f"  Max keywords: {keyword_train['num_keywords'].max()}")

In [ ]:
# Visualize keyword count distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(keyword_train['num_keywords'], bins=20, edgecolor='black', alpha=0.7, color='skyblue')
plt.xlabel('Number of Keywords')
plt.ylabel('Frequency')
plt.title('Distribution of Keyword Count')
plt.axvline(keyword_train['num_keywords'].mean(), color='red', linestyle='--', label='Mean')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(keyword_train['doc_length'], keyword_train['num_keywords'], alpha=0.5)
plt.xlabel('Document Length (words)')
plt.ylabel('Number of Keywords')
plt.title('Document Length vs Keyword Count')

plt.tight_layout()
plt.show()

In [ ]:
# Sample documents with keywords
print("Sample documents with keywords:\n")
samples = keyword_train.sample(3, random_state=42)

for idx, row in samples.iterrows():
    print(f"Document {idx + 1}:")
    print(f"Text: {row['document'][:200]}...")
    print(f"Keywords: {row['keywords_str']}")
    print(f"Count: {row['num_keywords']} keywords")
    print("-" * 80)
    print()

## 4. Data Quality Checks

In [ ]:
# Check for missing values
print("Missing values in sentiment dataset:")
print(sentiment_train.isnull().sum())
print("\nMissing values in keyword dataset:")
print(keyword_train.isnull().sum())

In [ ]:
# Check for duplicates
print(f"Duplicate texts in sentiment dataset: {sentiment_train['text'].duplicated().sum()}")
print(f"Duplicate documents in keyword dataset: {keyword_train['document'].duplicated().sum()}")

## 5. Summary Statistics

In [ ]:
summary = {
    "sentiment": {
        "total_samples": len(sentiment_train) + len(sentiment_val) + len(sentiment_test),
        "train_samples": len(sentiment_train),
        "val_samples": len(sentiment_val),
        "test_samples": len(sentiment_test),
        "classes": sentiment_train['label_name'].unique().tolist(),
        "avg_text_length": float(sentiment_train['text_length'].mean()),
        "class_distribution": sentiment_train['label_name'].value_counts().to_dict()
    },
    "keyword": {
        "total_samples": len(keyword_train) + len(keyword_val) + len(keyword_test),
        "train_samples": len(keyword_train),
        "val_samples": len(keyword_val),
        "test_samples": len(keyword_test),
        "avg_keywords": float(keyword_train['num_keywords'].mean()),
        "avg_doc_length": float(keyword_train['doc_length'].mean())
    }
}

print(json.dumps(summary, indent=2))

# Save summary
with open('../results/eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n✓ EDA summary saved to results/eda_summary.json")

## Conclusions

### Sentiment Dataset
- The dataset is relatively balanced across the three sentiment classes
- Average text length is suitable for BERT-based models (< 128 tokens)
- No significant missing values or data quality issues

### Keyword Dataset
- Documents vary in length with a good distribution of keywords
- Average keyword count is reasonable for seq2seq generation
- Clean dataset with minimal preprocessing needed

### Next Steps
1. Run baseline evaluation (baseline.py)
2. Fine-tune models with LoRA (lora_trainer.py)
3. Compare performance metrics